# Phishing URL classification: linear SVM grid search

**Nipun Yonjan**

This notebook tunes a linear SVM on the shared phishing-URL dataset. The preparation pipeline stays inside the grid search, so the scaler is refit on each training fold. The best model is scored on the dev split. The test split is left for the conclusion notebook.

## 1. Load the shared split and build the search

`prepare_datasets()` removes duplicate URLs and returns the same train, dev, and test split for the whole team. The raw training table goes into `GridSearchCV` without being scaled first. Scaling happens inside the pipeline on each fold.

The grid has four values of `C` and two class-weight settings, which covers both linear SVMs from the investigation: `C=1.0` with no class weighting, and `C=0.01` with balanced class weights. The search optimizes phishing F1.

In [1]:
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from dataset_preparation import build_preparation_pipeline, prepare_datasets

data = prepare_datasets()

print("Train rows:", len(data["y_train"]))
print("Dev rows:", len(data["y_dev"]))
print("Train phishing rate: {:.2%}".format(data["y_train"].mean()))
print("Dev phishing rate:   {:.2%}".format(data["y_dev"].mean()))
print("Columns going into the search:", data["X_train"].shape[1])

model = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual=False,
    max_iter=5000,
    random_state=42,
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

full_pipeline = Pipeline(
    steps=[
        ("prepare", build_preparation_pipeline()),
        ("model", model),
    ]
)

search = GridSearchCV(
    full_pipeline,
    param_grid,
    scoring="f1",
    cv=3,
)

print("Grid combinations:", len(ParameterGrid(param_grid)))
print("Search has been fit:", hasattr(search, "best_params_"))

Train rows: 69138
Dev rows: 23046
Train phishing rate: 14.40%
Dev phishing rate:   14.40%
Columns going into the search: 25
Grid combinations: 8
Search has been fit: False


The training split has the same 14.40% phishing rate as the dev split. Eight combinations will be scored with 3-fold cross-validation on phishing F1. The search has not been fit yet, and the dev and test splits are still unused.

## 2. Fit the grid search

Each of the 8 settings is scored with 3-fold cross-validation on the training split. The score is phishing F1. The dev split is not used to choose the winner.

In [2]:
import pandas as pd

search.fit(data["X_train"], data["y_train"])

print("Best parameters:", search.best_params_)
print("Best CV phishing F1: {:.4f}".format(search.best_score_))

results = pd.DataFrame(search.cv_results_)
results = results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score")
results

Best parameters: {'model__C': 10.0, 'model__class_weight': None}
Best CV phishing F1: 0.8178


,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
6,10.0,None,0.817796,0.002666,1
4,1.0,None,0.817618,0.002652,2
2,0.1,None,0.817064,0.002635,3
0,0.01,None,0.815993,0.002617,4
7,10.0,balanced,0.780798,0.002280,5
5,1.0,balanced,0.780798,0.002240,6
3,0.1,balanced,0.780523,0.002197,7
1,0.01,balanced,0.773289,0.001448,8


The best cross-validation result is `C=10.0` with no class weighting. Its mean phishing F1 is 0.8178.

The four unweighted settings all rank above the four balanced settings. Balanced weights were useful in the investigation because they raised recall, but this search is optimizing F1, and the unweighted models score higher on that metric. `C=10.0` is only slightly ahead of `C=1.0` (0.8176). The dev split has not been scored yet.

## 3. Score the best model on the dev split

`search.predict` uses the refit model from the winning settings, `C=10.0` with no class weighting. The dev rows still include the raw text columns, and the fitted pipeline drops and scales them. Phishing (`label = 1`) is the positive class. ROC AUC uses `decision_function` because `LinearSVC` has no `predict_proba`.

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_dev = data["y_dev"]
dev_pred = search.predict(data["X_dev"])
dev_score = search.decision_function(data["X_dev"])

dev_metrics = {
    "accuracy": accuracy_score(y_dev, dev_pred),
    "precision": precision_score(y_dev, dev_pred, pos_label=1),
    "recall": recall_score(y_dev, dev_pred, pos_label=1),
    "f1": f1_score(y_dev, dev_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_dev, dev_score),
}

print("Best linear SVM dev scores")
print("Accuracy:           {:.4f}".format(dev_metrics["accuracy"]))
print("Phishing precision: {:.4f}".format(dev_metrics["precision"]))
print("Phishing recall:    {:.4f}".format(dev_metrics["recall"]))
print("Phishing F1:        {:.4f}".format(dev_metrics["f1"]))
print("ROC AUC:            {:.4f}".format(dev_metrics["roc_auc"]))

Best linear SVM dev scores
Accuracy:           0.9511
Phishing precision: 0.9170
Phishing recall:    0.7260
Phishing F1:        0.8104
ROC AUC:            0.9760


On the dev set the tuned SVM reaches 95.11% accuracy, 0.810 phishing F1, and 0.976 ROC AUC. When it flags a URL, it is right 91.70% of the time, and it finds 72.60% of the phishing URLs.

Leaving class weights off is why precision is high and recall is lower. The search chose the setting with the best phishing F1, not the setting that catches the most phishing URLs. The test split is still unused.

## 4. Comparison with the investigation SVM

The investigation SVM used `C=0.01` and `class_weight="balanced"`. Its dev scores come from `Yonjan_Nipun_tree_svm_investigation.ipynb`. The tuned model is the grid-search winner, `C=10.0` with no class weighting. Both rows are dev scores. The test split is not included.

In [4]:
investigation_metrics = {
    "accuracy": 0.9226,
    "precision": 0.6669,
    "recall": 0.9244,
    "f1": 0.7748,
    "roc_auc": 0.9757,
}

comparison = pd.DataFrame(
    {
        "Investigation SVM": investigation_metrics,
        "Tuned SVM": dev_metrics,
    }
).T
comparison = comparison[["accuracy", "precision", "recall", "f1", "roc_auc"]]
comparison = comparison.rename(
    columns={
        "precision": "phishing_precision",
        "recall": "phishing_recall",
        "f1": "phishing_f1",
    }
)
comparison.round(4)

,accuracy,phishing_precision,phishing_recall,phishing_f1,roc_auc
Investigation SVM,0.9226,0.6669,0.9244,0.7748,0.9757
Tuned SVM,0.9511,0.9170,0.7260,0.8104,0.9760


## Conclusion

The tuned SVM is the model to keep. The search optimized phishing F1, and that score rises from 0.775 on the investigation SVM to 0.810 on the tuned SVM. Precision rises from 0.667 to 0.917, and accuracy rises from 92.26% to 95.11%. ROC AUC stays essentially the same, 0.976 against 0.976.

The tradeoff is recall. The balanced investigation SVM finds 92.44% of phishing URLs. The tuned SVM finds 72.60%. Turning class weights off creates fewer false alarms and misses more phishing sites.

For the group conclusion, this algorithm's selected model is the linear SVM with `C=10.0` and no class weighting, because the shared training plan chooses the model by dev phishing F1. The test split remains unused until `conclusion.ipynb`.